<a href="https://colab.research.google.com/github/mobadara/precision-diagnostics-xai/blob/main/notebooks/01_model_training_and_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Model Training and Fine-Tuning**

**Objective:** Ingest the Kaggle Chest X-Ray dataset, apply strategic data augmentation, handle severe class imbalance via loss weighting, and fine-tune a pre-trained DenseNet121 architecture to detect pneumonia.

## **Environment Setup**
First, we import the necessary PyTorch libraries and configure our hardware accelerator. Because image processing is computationally expensive, we must ensure PyTorch detects and utilizes the Colab GPU (CUDA).

In [5]:
%%capture install_output
!pip install optuna

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
import optuna
import time
import copy
import numpy as np

# Configure the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute Device: {device}')
if device.type == 'cuda':
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')

Compute Device: cuda
GPU Model: Tesla T4


## **Data Preprocessing and Augmentation Pipeline**
As determined in our EDA, we must standardize all input resolutions to 224x224.

To prevent overfitting without introducing artificial artifacts, we apply light data augmentation **only to the training set** (rotation and color jitter). The validation and test sets remain strictly unaltered to provide an honest evaluation metric.

In [2]:
# Define the dataset paths
data_dir = '../dataset/chest_xray'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

# 1. Training Transforms (with Augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10), # Slight rotation to simulate patient misalignment
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Account for different X-ray exposures
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet normalization
])

# 2. Validation/Testing Transforms (Strictly unaltered)
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Transforms configured successfully.")

Transforms configured successfully.


## **Initializing Datasets and DataLoaders**

With our transforms defined, we will use PyTorch's `ImageFolder` to automatically read our directory structure and assign the correct labels (0 for Normal, 1 for Pneumonia).

We will then wrap these datasets in `DataLoaders`. DataLoaders handle the heavy lifting of batching the images, shuffling them, and pushing them to the GPU. We are setting the `batch_size` to 64 to fully utilize the Google Colab GPU memory.

In [3]:
# 1. Create the Dataset objects
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=eval_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=eval_transforms)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Testing images: {len(test_dataset)}")
print(f"Detected Classes: {train_dataset.classes}")

# 2. Create the DataLoaders
BATCH_SIZE = 64

# We shuffle the training data so the model doesn't learn sequence patterns
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

# We do not need to shuffle validation/testing data
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("DataLoaders are ready!")

Training images: 5216
Validation images: 16
Testing images: 624
Detected Classes: ['NORMAL', 'PNEUMONIA']
DataLoaders are ready!


## **Model Architecture and Class Weighting**

Instead of training a model from scratch, we use **Transfer Learning**. We will download DenseNet121 (which already knows how to detect edges, textures, and shapes from millions of standard images) and replace its final classification layer with a custom one designed specifically for our two classes (Normal vs. Pneumonia).

Simultaneously, we will calculate the inverse frequency of our classes to generate **Class Weights**, injecting them directly into the PyTorch loss function to penalize the model more heavily if it misclassifies a minority "Normal" scan.

In [7]:
# --- 1. Handling the Imbalance (Class Weights) ---
# ImageFolder automatically stores the label for every image in the .targets attribute
labels = train_dataset.targets
class_counts = np.bincount(labels)
total_samples = len(labels)
num_classes = len(train_dataset.classes)

# Inverse frequency formula: Weight = Total / (NumClasses * ClassCount)
weights = total_samples / (num_classes * class_counts)
class_weights = torch.FloatTensor(weights).to(device)

print(f"Class Counts: NORMAL={class_counts[0]}, PNEUMONIA={class_counts[1]}")
print(f"Applied Weights: NORMAL={weights[0]:.4f}, PNEUMONIA={weights[1]:.4f}\n")

# --- 2. Model Architecture (DenseNet121) ---
# Download the pre-trained neural network
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

# Freeze the core convolutional layers so we don't destroy the pre-trained visual features
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer (originally 1000 classes) with our custom 2-class head
num_ftrs = model.classifier.in_features

# Push the entire model architecture to the GPU
model = model.to(device)

# --- 3. Loss Function and Optimizer ---
# Inject our calculated class weights into the loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)

# We only want the optimizer to update our newly added classifier layers
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print("Model architecture, optimizer, and weighted loss function configured successfully!")

Class Counts: NORMAL=1341, PNEUMONIA=3875
Applied Weights: NORMAL=1.9448, PNEUMONIA=0.6730

Model architecture, optimizer, and weighted loss function configured successfully!
